In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv("../data/processed/hr_clean.csv")

# Create (or open) a database file in the sql folder
conn = sqlite3.connect("../sql/hr.db")

# Copy the DataFrame into a database table called employees
df.to_sql("employees", conn, index=False, if_exists="replace")

1470

In [2]:
def q(sql):
    return pd.read_sql(sql, conn)

In [3]:
q("""
SELECT COUNT(*) AS headcount,
       SUM(AttritionFlag) AS leavers,
       ROUND(100.0 * SUM(AttritionFlag) / COUNT(*), 1) AS attrition_pct
FROM employees
""")

,headcount,leavers,attrition_pct
0,1470,237,16.1


In [4]:
q("""
SELECT JobRole,
       COUNT(*) AS headcount,
       SUM(AttritionFlag) AS leavers,
       ROUND(100.0 * SUM(AttritionFlag) / COUNT(*), 1) AS attrition_pct
FROM employees
GROUP BY JobRole
HAVING COUNT(*) >= 50
ORDER BY attrition_pct DESC
""")

,JobRole,headcount,leavers,attrition_pct
0,Sales Representative,83,33,39.8
1,Laboratory Technician,259,62,23.9
2,Human Resources,52,12,23.1
3,Sales Executive,326,57,17.5
4,Research Scientist,292,47,16.1
5,Manufacturing Director,145,10,6.9
6,Healthcare Representative,131,9,6.9
7,Manager,102,5,4.9
8,Research Director,80,2,2.5


In [5]:
q("""
SELECT OverTime, AgeBand,
       COUNT(*) AS headcount,
       ROUND(100.0 * AVG(AttritionFlag), 1) AS attrition_pct
FROM employees
GROUP BY OverTime, AgeBand
ORDER BY attrition_pct DESC
""")

,OverTime,AgeBand,headcount,attrition_pct
0,Yes,18-25,39,64.1
1,Yes,26-35,159,39.0
2,No,18-25,84,22.6
3,Yes,46-60,83,20.5
4,Yes,36-45,135,17.0
5,No,26-35,447,12.1
6,No,46-60,190,8.9
7,No,36-45,333,6.0


In [6]:
q("""
SELECT Department,
       COUNT(*) AS headcount,
       ROUND(100.0 * AVG(AttritionFlag), 1) AS attrition_pct
FROM employees
WHERE OverTime = 'Yes' AND IncomeBand = 'Q1 Lowest'
GROUP BY Department
ORDER BY attrition_pct DESC
""")

,Department,headcount,attrition_pct
0,Sales,20,70.0
1,Human Resources,6,66.7
2,Research & Development,80,55.0
